# Pooled Classification

A common workflow with longitudinal spatial data is to apply the same classification scheme to an attribute over different time periods. More specifically, one would like to keep the class breaks the same over each period and examine how the mass of the distribution changes over these classes in the different periods.

The `Pooled` classifier supports this workflow.

In [ ]:
import numpy

import mapclassify

mapclassify.__version__

## Sample Data
We construct a synthetic dataset composed of 20 cross-sectional units at three time points. Here the mean of the series is increasing over time.

In [ ]:
n = 20
data = numpy.array([numpy.arange(n) + i * n for i in range(1, 4)]).T
data.shape

In [ ]:
data

## Default: Quintiles
The default is to apply a [vec](https://en.wikipedia.org/wiki/Vectorization_(mathematics)) operator to the data matrix and treat the observations as a single collection. Here the quantiles of the pooled data are obtained.

In [ ]:
res = mapclassify.Pooled(data)
res

Note that the class definitions are constant across the periods.

In [ ]:
res = mapclassify.Pooled(data, k=4)

In [ ]:
res.col_classifiers[0].counts

In [ ]:
res.col_classifiers[-1].counts

In [ ]:
res.global_classifier.counts

In [ ]:
res

Extract the pooled classification objects for each column.

In [ ]:
c0, c1, c2 = res.col_classifiers
c0

Compare to the unrestricted classifier for the first column...

In [ ]:
mapclassify.Quantiles(c0.y, k=4)

... and the last column comparisions.

In [ ]:
c2

In [ ]:
mapclassify.Quantiles(c2.y, k=4)

## Non-default classifier: BoxPlot

In [ ]:
res = mapclassify.Pooled(data, classifier="BoxPlot", hinge=1.5)
res

In [ ]:
res.col_classifiers[0].bins

In [ ]:
c0, c1, c2 = res.col_classifiers

In [ ]:
c0.yb

In [ ]:
c00 = mapclassify.BoxPlot(c0.y, hinge=3)

In [ ]:
c00.yb

In [ ]:
c00

In [ ]:
c0

## Non-default classifier: FisherJenks

In [ ]:
res = mapclassify.Pooled(data, classifier="FisherJenks", k=5)
res

In [ ]:
c0, c1, c2 = res.col_classifiers
mapclassify.FisherJenks(c0.y, k=5)

## Non-default classifier: MaximumBreaks



In [ ]:
data[1, 0] = 10
data[1, 1] = 10
data[1, 2] = 10
data[9, 2] = 10
data

In [ ]:
res = mapclassify.Pooled(data, classifier="MaximumBreaks", k=5)
res

In [ ]:
c0, c1, c2 = res.col_classifiers
c0

In [ ]:
c0.y

In [ ]:
import warnings

In [ ]:
with warnings.catch_warnings():
    warnings.filterwarnings("error")
    try:
        mapclassify.MaximumBreaks(c0.y, k=5)
    except UserWarning as e:
        print(e)
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore")
            mapclassify.MaximumBreaks(c0.y, k=5)

In [ ]:
res = mapclassify.Pooled(
    data, classifier="UserDefined", bins=mapclassify.Quantiles(data[:, -1]).bins
)
res

In [ ]:
mapclassify.Quantiles(data[:, -1])

In [ ]:
data[:, -1]

## Pinning the pooling

Another option is to specify a specific subperiod as the definition for the classes in the pooling.

### Pinning to the last period

As an example, we can use the quintles from the third period to defined the pooled classifier:

In [ ]:
pinned = mapclassify.Pooled(
    data, classifier="UserDefined", bins=mapclassify.Quantiles(data[:, -1]).bins
)
pinned

In [ ]:
pinned.global_classifier

### Pinning to the first period

In [ ]:
pinned = mapclassify.Pooled(
    data, classifier="UserDefined", bins=mapclassify.Quantiles(data[:, 0]).bins
)
pinned

Note that the quintiles for the first period, by definition, contain all the values from that period, they do not bound the larger values in subsequent periods. Following the [mapclassify policy](https://github.com/pysal/mapclassify/blob/a7770fb98bf945dad3c62ccf2c0f8b53abb1774a/mapclassify/classifiers.py#L589), an additional class is added to contain all values in the pooled series.